## Week_5 Assignments
 - A code tool that automatically adds docstring / comments
 - A code gen tool that writes unit test cases
 - A code generator that writes trading code to buy and sell equities in a simulated environment, based on a given API

In [ ]:
import os
import sys
import gradio as gr
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
MODEL_GO120 = "openai/gpt-oss-120b"

In [ ]:
#import api key from environment variable
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openrouter_api_key and openrouter_api_key.startswith("sk-or-"):
    print("OPENROUTER_API_KEY is set correctly.")

In [ ]:
openrouter = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

In [ ]:
#test the model
response = openrouter.chat.completions.create(
    model=MODEL_GO120,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is the capital of France?"}
    ]
)

display(Markdown(response.choices[0].message.content))

In [ ]:
system_prompt = """
You are a senior software engineer with expertise in Python and C++. 
You have a strong background in software development, code optimization, and debugging. 
Your task is to convert code snippets from Python to C++ while maintaining the same functionality and logic. 
You should ensure that the converted code is efficient, follows best practices, and is easy to understand. 
Pay attention to details such as variable naming conventions, data types, and error handling. 
Your goal is to produce high-quality C++ code that accurately reflects the original Python code with high efficiency.
"""

In [ ]:
def user_prompt(python_code):
    return f"""
    Convert the following Python code to C++. 
    Make sure to pay attention to the details such as variable naming conventions, data types, and error handling. 
    Your response will be written to a file called main.cpp and then compiled and executed. 
    Respond only with C++ code.
    Here's is the code:\n\n{python_code}
    """

In [ ]:
def save_cpp_code(cpp_code):
    with open("main.cpp", "w") as f:
        f.write(cpp_code)

In [ ]:
def convert_python_to_cpp(python_code):
    print("[DEBUG]Converting Python code to C++...")
    response = openrouter.chat.completions.create(
        model=MODEL_GO120,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt(python_code)}
        ]
    )
    print("[DEBUG]Received response from model.")
    cpp_code = response.choices[0].message.content

    save_cpp_code(cpp_code)
    return cpp_code

In [ ]:
def run_python_code(python_code):
    print("[DEBUG]Running Python code...")
    globals = {"__builtins__": __builtins__}
    exec(python_code, globals)

In [ ]:
def run_cpp_code(cpp_code):
    print("[DEBUG]Saving C++ code to main.cpp...")
    with open("main.cpp", "w") as f:
        f.write(cpp_code)
    
    print("[DEBUG]Compiling the C++ code...")
    os.system("g++ main.cpp -o main")
    print("[DEBUG]Running the compiled C++ code...")
    os.system("./main.exe")

In [ ]:
with gr.Blocks() as demo:
    python_input = gr.Code(language="python", label="Enter Python code here...")
    cpp_output = gr.Code(language="cpp", label="C++ code will appear here...")

    gr.Markdown("## Python to C++ Code Converter")
    gr.Markdown("Enter your Python code and get the equivalent C++ code.")

    convert_button = gr.Button("Convert to C++")
    run_python_button = gr.Button("Run Python Code", variant="secondary")
    run_cpp_button = gr.Button("Run C++ Code", variant="secondary")

    convert_button.click(
        fn=convert_python_to_cpp,
        inputs=python_input,
        outputs=cpp_output,
    )
    run_python_button.click(fn=run_python_code, inputs=python_input, outputs=None)
    run_cpp_button.click(fn=run_cpp_code, inputs=cpp_output, outputs=None)

demo.launch(server_name="0.0.0.0")